In [3]:
import torch


class DynamicMask:
    def __init__(self, min_alpha=0.1, max_alpha=0.9, smoothing_factor=0.9):
        self.min_alpha = min_alpha
        self.max_alpha = max_alpha
        self.smoothing_factor = smoothing_factor
        self.prev_alpha = None

    def mask_topk(self, x, largest=False):
        # B: batch size, H: number of heads, L: sequence length
        B, H, L, _ = x.shape

        # 动态计算每个头的 alpha
        alpha_per_head = []
        for b in range(B):
            alpha_list = []
            for h in range(H):
                # 根据数据的标准差动态调整 alpha
                std = torch.std(x[b, h])
                alpha = (std - torch.min(x)) / (torch.max(x) - torch.min(x))
                alpha = alpha.item()
                alpha = max(self.min_alpha, min(self.max_alpha, alpha))
                if self.prev_alpha is not None:
                    alpha = self.smoothing_factor * self.prev_alpha + (1 - self.smoothing_factor) * alpha
                alpha_list.append(alpha)
            alpha_per_head.append(alpha_list)
        alpha_per_head = torch.tensor(alpha_per_head, device=x.device)

        mask = torch.zeros_like(x, dtype=torch.float32)
        for b in range(B):
            for h in range(H):
                alpha = alpha_per_head[b, h]
                k = int(alpha * L)
                _, topk_indices = torch.topk(x[b, h], k, dim=-1, largest=largest)
                head_mask = torch.ones_like(x[b, h], dtype=torch.float32)
                head_mask.scatter_(-1, topk_indices, 0)
                mask[b, h] = head_mask

        self.prev_alpha = alpha_per_head.mean()
        return mask

# 创建实例
dynamic_mask = DynamicMask()

# 示例输入
x = torch.randn(2, 4, 10, 10)  # [B, H, L, L]

# 生成掩码
mask = dynamic_mask.mask_topk(x)
print(mask.shape) 

torch.Size([2, 4, 10, 10])


In [4]:
import torch


class DynamicMask:
    def __init__(self, min_alpha=0.1, max_alpha=0.9, smoothing_factor=0.9):
        self.min_alpha = min_alpha
        self.max_alpha = max_alpha
        self.smoothing_factor = smoothing_factor
        self.prev_alpha = None

    def mask_topk(self, x, largest=False):
        # B: batch size, H: number of heads, L: sequence length
        B, H, L, _ = x.shape

        # 动态计算每个头的 alpha
        alpha_per_head = []
        for b in range(B):
            alpha_list = []
            for h in range(H):
                # 根据数据的标准差动态调整 alpha
                std = torch.std(x[b, h])
                alpha = (std - torch.min(x)) / (torch.max(x) - torch.min(x))
                alpha = alpha.item()
                alpha = max(self.min_alpha, min(self.max_alpha, alpha))
                if self.prev_alpha is not None:
                    alpha = self.smoothing_factor * self.prev_alpha + (1 - self.smoothing_factor) * alpha
                alpha_list.append(alpha)
            alpha_per_head.append(alpha_list)
        alpha_per_head = torch.tensor(alpha_per_head, device=x.device)

        mask = torch.zeros_like(x, dtype=torch.float32)
        for b in range(B):
            for h in range(H):
                alpha = alpha_per_head[b, h]
                k = int(alpha * L)
                _, topk_indices = torch.topk(x[b, h], k, dim=-1, largest=largest)
                head_mask = torch.ones_like(x[b, h], dtype=torch.float32)
                head_mask.scatter_(-1, topk_indices, 0)
                mask[b, h] = head_mask

        self.prev_alpha = alpha_per_head.mean()
        return mask


# 示例相关性矩阵，形状为 [32, 4, 42, 42]
correlation_matrix = torch.randn(32, 4, 42, 42)

# 创建 DynamicMask 类的实例
dynamic_mask = DynamicMask()

# 调用 mask_topk 方法获取掩码矩阵
mask_matrix = dynamic_mask.mask_topk(correlation_matrix)

print("掩码矩阵的形状:", mask_matrix.shape)

掩码矩阵的形状: torch.Size([32, 4, 42, 42])


In [ ]:
class mask_moe(nn.Module):
    def __init__(self, n_vars, top_p=0.5, num_experts=3, in_dim=96):
        super().__init__()
        self.num_experts = num_experts
        self.n_vars = n_vars
        self.in_dim = in_dim
        self.top_p = top_p

        self.gate = nn.Linear(in_dim, num_experts, bias=False)
        self.noise = nn.Linear(in_dim, num_experts, bias=False)
        self.noisy_gating = True
        self.softplus = nn.Softplus()
        self.softmax = nn.Softmax(2)
        
        # 新增：可学习的专家注意力层（方法2）
        self.expert_attn = nn.ModuleList([
            nn.Linear(in_dim, 1) for _ in range(num_experts)
        ])

    # ...（保留cv_squared、cross_entropy、noisy_top_k_gating方法）

    def forward(self, x, is_training=None):  # 移除masks参数
        B, H, L, _ = x.shape
        device = x.device
        dtype = torch.float32
        mask_base = torch.eye(L, device=device, dtype=dtype).unsqueeze(0).unsqueeze(0)

        if self.top_p == 0.0:
            return mask_base, 0.0

        # 动态生成专家掩码（两种方法任选其一）
        masks = self.dynamic_generate_masks(x, L, device, dtype)  # [L, 3, L]

        x = x.reshape(B * H, L, self.in_dim)  # 假设x为[B, H, L, in_dim]
        gates, loss = self.noisy_top_k_gating(x, is_training)
        gates = gates.reshape(B, H, L, -1).float()  # [B, H, L, 3]

        mask = torch.einsum('bhli,lid->bhld', gates, masks) + mask_base
        return mask, loss

    def dynamic_generate_masks(self, x, L, device, dtype):
        """动态生成专家掩码的两种方式"""
        B, H, _, in_dim = x.shape
        x = x.reshape(B*H, L, in_dim)  # [B*H, L, in_dim]
        
        ####################
        # 方法1：基于相似度的掩码
        ####################
        # 计算节点间相似度（点积）
        # x_feats = x.mean(dim=0)  # 全局平均去批次 [L, in_dim]
        x_feats = x  # 保留批次信息
        sim = torch.matmul(x_feats, x_feats.transpose(1, 2))  # [B*H, L, L]
        sim = sim.softmax(dim=-1)  # 归一化相似度

        ####################
        # 方法2：可学习的注意力掩码
        ####################
        sim = []
        for e in range(self.num_experts):
            # 专家e的注意力分数：(x_i - x_j)的绝对值加权
            delta = torch.abs(x.unsqueeze(2) - x.unsqueeze(1))  # [B*H, L, L, in_dim]
            attn = self.expert_attn[e](delta.reshape(-1, in_dim)).reshape(B*H, L, L)
            sim_e = attn.softmax(dim=-1)
            sim.append(sim_e)
        sim = torch.stack(sim, dim=2)  # [B*H, L, 3, L]

        ####################
        # 生成Top-p掩码
        ####################
        masks = []
        for batch in range(B*H):
            for node in range(L):
                expert_masks = []
                for e in range(self.num_experts):
                    probs = sim[batch, node, e, :]  # 专家e的概率分布
                    sorted_probs, sorted_idx = torch.sort(probs, descending=True)
                    cumulative = torch.cumsum(sorted_probs, dim=0)
                    mask = cumulative > self.top_p  # 保留累积概率>top_p的边
                    mask = torch.zeros_like(probs).scatter_(0, sorted_idx, mask)
                    expert_masks.append(mask)
                masks.append(torch.stack(expert_masks, dim=0))  # [3, L]
        masks = torch.stack(masks, dim=0).reshape(B*H, self.num_experts, L)
        masks = masks.mean(dim=0).to(device=device, dtype=dtype)  # [3, L] -> 全局平均
        return masks.permute(1, 0, 2)  # [L, 3, L]